<a href="https://colab.research.google.com/github/ARCHITTOMAR15/Sarcasm-Detection-with-Traditional-NLP-Deep-Learning-and-Transformers/blob/main/SARCSAM_DETECTION_USING_TRANSFORMER_MODELS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.listdir('/content/drive/MyDrive/sarcsam_detection')

['Sarcasm_Headlines_Dataset_v2.json']

In [3]:
from datasets import Dataset

In [4]:
# import all necessary libraries
import re
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [5]:
#data reading
df=pd.read_json('/content/drive/MyDrive/sarcsam_detection/Sarcasm_Headlines_Dataset_v2.json',lines=True)

In [6]:
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28619 entries, 0 to 28618
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   is_sarcastic  28619 non-null  int64 
 1   headline      28619 non-null  object
 2   article_link  28619 non-null  object
dtypes: int64(1), object(2)
memory usage: 670.9+ KB


In [8]:
df.isnull().sum()

,0
is_sarcastic,0
headline,0
article_link,0


In [9]:
df.duplicated().sum()

np.int64(2)

In [10]:
df["headline"].duplicated().sum()

np.int64(116)

In [11]:
df=df.drop_duplicates(subset=["headline"]).reset_index(drop=True)

In [12]:
df["headline"].duplicated().sum()

np.int64(0)

In [13]:
# Basic minimal clearning
def clean_text(text):
  text=str(text)
  # remove URLs
  text = re.sub(r"http\S+|www\S+", "", text)
  # remove extra spaces
  text = re.sub(r"\s+", " ", text)

  return text.strip()

In [14]:
df["clean_headline"]=df["headline"].apply(clean_text)

In [15]:
df["is_sarcastic"].value_counts()

,count
is_sarcastic,
0,14951
1,13552


In [16]:
df["is_sarcastic"].value_counts(normalize=True)

,proportion
is_sarcastic,
0,0.524541
1,0.475459


In [17]:
#       Create train,test and validation slpit

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train_df,temp_df=train_test_split(df,test_size=0.20,random_state=42,stratify=df["is_sarcastic"])

In [20]:
# split temp_df into test and validation
val_df,test_df=train_test_split(temp_df,test_size=0.50,random_state=42,stratify=temp_df["is_sarcastic"])

In [21]:
#reset index
train_df=train_df.reset_index(drop=True)
test_df=test_df.reset_index(drop=True)
val_df=val_df.reset_index(drop=True)

In [22]:
print("Train_shape",train_df.shape)
print("Test_shape",test_df.shape)
print("val_shape",val_df.shape)

Train_shape (22802, 4)
Test_shape (2851, 4)
val_shape (2850, 4)


In [23]:
# convert pandas dataframe to hugging face dataset
from datasets import Dataset

In [24]:
train_dataset=Dataset.from_pandas(train_df[["clean_headline","is_sarcastic"]])
test_dataset=Dataset.from_pandas(test_df[["clean_headline","is_sarcastic"]])
val_dataset=Dataset.from_pandas(val_df[["clean_headline","is_sarcastic"]])

In [25]:
# rename is_sarcastc to label
train_dataset=train_dataset.rename_column("is_sarcastic","label")
test_dataset=test_dataset.rename_column("is_sarcastic","label")
val_dataset=val_dataset.rename_column("is_sarcastic","label")

**MODEL 1 : BERT BASIC UNCASED**

In [ ]:
#### IMPORTING TOKENIZER
from transformers import DistilBertTokenizerFast
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [ ]:
print(tokenizer)

In [ ]:
# Tokenise the Entire Data set (Train,test,validation data set) create a function

def tokenize(batch):
  return tokenizer(batch["clean_headline"],padding='max_length',truncation=True,max_length=128)


In [ ]:
train_dataset =train_dataset.map(tokenize,batched=True)
test_dataset=test_dataset.map(tokenize,batched=True)
val_dataset=val_dataset.map(tokenize,batched=True)

In [ ]:
print(train_dataset[0])

In [ ]:
#  Convert to PYtorch Format for Batch used in Training
train_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])
test_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])
val_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])

In [ ]:
# Create Data loaders
BATCH_SIZE=16
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
batch=next(iter(train_loader))
print(batch.keys())

In [ ]:
# LOAD PRETRAINED DISTILBERT MODEL

In [ ]:
from transformers import DistilBertModel
distilbert=DistilBertModel.from_pretrained("distilbert-base-uncased")

In [ ]:
# PART 2

In [ ]:
## FREEZE THE ORIZINAL DISTILBERT PARAMETERS
for param in distilbert.parameters():
  param.requires_grad=False

In [ ]:
print(distilbert)

In [ ]:
## Making own classififer head
import torch
import torch.nn as nn

In [ ]:
class DistilBERTClassifier(nn.Module):

    def __init__(self, distilbert):

        super().__init__()

        self.distilbert = distilbert

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0]

        x = self.dropout(cls_embedding)

        logits = self.classifier(x)

        return logits

In [ ]:
model = DistilBERTClassifier(distilbert)

In [ ]:
#Defining the loss function

In [ ]:
criterion=nn.CrossEntropyLoss()

In [ ]:
#optimizer and schedular

In [ ]:
from torch.optim import AdamW

In [ ]:
optimizer=AdamW(model.parameters(),lr=0.0002)

In [ ]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
EPOCHS=5
total_training_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps
)

In [ ]:
## PART - 3

In [ ]:
## CREATE TRAINING LOOP

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

In [ ]:
train_losses = []
train_accuracies = []

for epoch in range(EPOCHS):

    # -----------------------------
    # Training Mode
    # -----------------------------
    model.train()

    running_loss = 0
    correct_predictions = 0
    total_examples = 0

    # Iterate over batches
    for batch in train_loader:

        # Move tensors to GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Compute loss
        loss = criterion(logits, labels)

        # Backpropagation
        loss.backward()

        # Update trainable parameters
        optimizer.step()

        # Update learning rate
        scheduler.step()

        # Store batch loss
        running_loss += loss.item()

        # Predictions
        predictions = torch.argmax(logits, dim=1)

        # Count correct predictions
        correct_predictions += (predictions == labels).sum().item()

        # Total samples
        total_examples += labels.size(0)

    # Epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_examples

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Training Loss : {epoch_loss:.4f}")
    print(f"Training Accuracy : {epoch_accuracy:.4f}")

In [ ]:
## Validation CODE

In [ ]:
val_losses = []
val_accuracies = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0
    correct_predictions = 0
    total_examples = 0

    # -----------------------------
    # TRAINING
    # -----------------------------
    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        scheduler.step()

        running_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)

        correct_predictions += (predictions == labels).sum().item()

        total_examples += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct_predictions / total_examples

    # -----------------------------
    # VALIDATION
    # -----------------------------
    model.eval()

    running_val_loss = 0
    correct_predictions = 0
    total_examples = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)

            running_val_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)

            correct_predictions += (predictions == labels).sum().item()

            total_examples += labels.size(0)

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = correct_predictions / total_examples

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print("-"*50)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_accuracy:.4f}")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Acc. : {val_accuracy:.4f}")

In [ ]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

In [ ]:
model.eval()

test_loss = 0

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)

        test_loss += loss.item()

        probabilities = torch.softmax(logits, dim=1)

        predictions = torch.argmax(probabilities, dim=1)

        all_labels.extend(labels.cpu().numpy())

        all_predictions.extend(predictions.cpu().numpy())

        all_probabilities.extend(
            probabilities[:,1].cpu().numpy()
        )

test_loss = test_loss / len(test_loader)

print(f"Test Loss : {test_loss:.4f}")

In [ ]:
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

In [ ]:
precision = precision_score(
    all_labels,
    all_predictions
)

In [ ]:
recall = recall_score(
    all_labels,
    all_predictions
)

In [ ]:
f1 = f1_score(
    all_labels,
    all_predictions
)

In [ ]:
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Sarcastic","Sarcastic"],
    yticklabels=["Not Sarcastic","Sarcastic"]
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Not Sarcastic",
            "Sarcastic"
        ]
    )
)

In [ ]:
auc = roc_auc_score(
    all_labels,
    all_probabilities
)

print(f"ROC AUC : {auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(
    all_labels,
    all_probabilities
)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    "--"
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Training Loss")

plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_accuracies,
    label="Training Accuracy"
)

plt.plot(
    val_accuracies,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend()

plt.show()

## **MODEL 2  :DISTILBERT CLASSIFICATION FOR SEQUENCE USING HUGGING FACE PIPELINE**




In [26]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [27]:
model_name= "helinivan/english-sarcasm-detector"

In [28]:
#Load Tokenizer
tokenizer1=AutoTokenizer.from_pretrained(model_name)

In [29]:
#Load Model
model1=AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# lets tokenize the data set

In [31]:
def tokenize_function(batch):
  return tokenizer1(batch["clean_headline"],
                    padding='max_length',
                    truncation=True,
                    max_length=128)

In [32]:
# lets map this tikenize to train tex and val function

In [33]:
train_dataset1=train_dataset.map(tokenize_function,batched=True)
test_dataset1=test_dataset.map(tokenize_function,batched=True)
val_dataset1=val_dataset.map(tokenize_function,batched=True)

Map:   0%|          | 0/22802 [00:00<?, ? examples/s]

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

In [34]:
print(train_dataset1.column_names)

['clean_headline', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [35]:
print(train_dataset1[0])

{'clean_headline': "'gobbler games' is the brutal hunger games parody you need to see", 'label': 0, 'input_ids': [101, 1005, 2175, 11362, 2099, 2399, 1005, 2003, 1996, 12077, 9012, 2399, 12354, 2017, 2342, 2000, 2156, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1,

In [36]:
## CONVERT THE TOKENIZE DATASET INTO PYTORCH FORMATTING

In [37]:
train_dataset1.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])
test_dataset1.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])
val_dataset1.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [39]:
print(type(train_dataset1[0]["input_ids"]))
print(type(train_dataset1[0]["attention_mask"]))
print(type(train_dataset1[0]["label"]))

<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [40]:
# lets create a inference code from the pretrained model

In [41]:
import torch

model1.eval()

predictions=[]
true_label=[]

# disable gradient computation
with torch.no_grad():
  for sample in test_dataset1:
    input_ids=sample["input_ids"].unsqueeze(0)
    attention_mask=sample["attention_mask"].unsqueeze(0)

    #forword pass
    outputs=model1(input_ids=input_ids,attention_mask=attention_mask)
    #get predicted class
    predicted_class=torch.argmax(outputs.logits,dim=1).item()

    #store prediction and actual label
    predictions.append(predicted_class)
    true_label.append(sample["label"].item())


In [42]:
for i in range(10):
  print(f"Actual:{true_label[i]}|Predicted{predictions[i]}")

Actual:0|Predicted0
Actual:1|Predicted1
Actual:0|Predicted0
Actual:1|Predicted1
Actual:0|Predicted0
Actual:0|Predicted0
Actual:1|Predicted1
Actual:1|Predicted1
Actual:1|Predicted1
Actual:1|Predicted1


In [43]:
#### LETS EVALUATE THE MODEL1 WHICH IS PRERRAINED

In [44]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score)

In [45]:
# Accuracy
accuracy=accuracy_score(true_label,predictions)
precision=precision_score(true_label,predictions)
recall=recall_score(true_label,predictions)



In [46]:
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)


Accuracy : 0.9400210452472817
Precision: 0.9618082618862042
Recall   : 0.9100294985250738


In [47]:
from sklearn.metrics import classification_report as cls_report
print(cls_report(true_label,predictions,target_names=["Not Sarcastic", "Sarcastic"]))

               precision    recall  f1-score   support

Not Sarcastic       0.92      0.97      0.94      1495
    Sarcastic       0.96      0.91      0.94      1356

     accuracy                           0.94      2851
    macro avg       0.94      0.94      0.94      2851
 weighted avg       0.94      0.94      0.94      2851



In [48]:
from sklearn.metrics import confusion_matrix as cmt
print(cmt(true_label,predictions))

[[1446   49]
 [ 122 1234]]


In [49]:
print(model1.config.id2label)
print(model1.config.label2id)

{0: 'LABEL_0', 1: 'LABEL_1'}
{'LABEL_0': 0, 'LABEL_1': 1}


##**MODEL 3 : DISTILLBERT WITH LORA FINETUNING**

In [50]:
# Load Distilber bert tokenizer forst and tokenize the datset

In [51]:
from transformers import AutoTokenizer

In [52]:
tokenizer3=AutoTokenizer.from_pretrained('distilbert-base-uncased')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [53]:
print(type(tokenizer3))

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


In [54]:
# create tokenization function
def tokenize_function(examples):
  return tokenizer3(examples["clean_headline"],turncation=True,max_length=128)



In [55]:
#Tokenizer all three dataset

In [56]:
train_tokenized2=train_dataset.map(tokenize_function)
test_tokenized2=test_dataset.map(tokenize_function)
val_tokenized2=val_dataset.map(tokenize_function)

Map:   0%|          | 0/22802 [00:00<?, ? examples/s]

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

In [57]:
print(train_tokenized2)

Dataset({
    features: ['clean_headline', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 22802
})


In [58]:
# lets try Dynamic Padding this time

In [59]:
from transformers import DataCollatorWithPadding

In [60]:
data_collator=DataCollatorWithPadding(tokenizer=tokenizer3)

In [61]:
## Model BUILDING  WITH LORA ADAPTERS

In [62]:
from transformers import AutoModelForSequenceClassification

In [63]:
model3=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [64]:
# LORA CINFIGURATION

In [65]:
from peft import LoraConfig,get_peft_model

In [66]:
lora_config=LoraConfig(r=8,lora_alpha=16,target_modules=["q_lin","v_lin"],lora_dropout=0.1,bias="none",task_type="SEQ_CLS")

In [67]:
# COMBINE DistilBert + LORA

In [69]:
!pip uninstall -y torchao
!pip install -U "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.7 MB/s eta 0:00:00


In [70]:
lora_model=get_peft_model(model3,lora_config)

W0730 05:59:28.481000 7054 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [71]:
print(lora_model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
               

In [72]:
# lets check trainable parameter after inserting lora

In [73]:
lora_model.print_trainable_parameters()

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [74]:
for name,param in lora_model.named_parameters():
  if param.requires_grad:
    print(name)

base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.2.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.2.attent

In [75]:
## DEFININF METRICS

In [76]:
from sklearn.metrics import ( accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix)

In [77]:
def compute_metrics(eval_pred):
  logits,labels=eval_pred
  predictions=np.argmax(logits,axis=-1)

  accuracy=accuracy_score(labels,predictions)
  precision=precision_score(labels,predictions)
  recall=recall_score(labels,predictions)
  f1=f1_score(labels,predictions)

  return {"accuracy":accuracy,"precision":precision,"recall":recall,"f1":f1}

In [78]:
## TRAINIG ARGUMENTS
from transformers import TrainingArguments

In [79]:
training_args=TrainingArguments(output_dir="/content/drive/MyDrive/distilbert_lora_sarcasm",
                                learning_rate=0.00002,
                                per_device_train_batch_size=16,
                                per_device_eval_batch_size=16,
                                num_train_epochs=3,
                                weight_decay=0.01,
                                eval_strategy="epoch",
                                save_strategy="epoch",
                                load_best_model_at_end=True,
                                metric_for_best_model="f1",
                                greater_is_better=True,
                                logging_steps=100,
                                report_to="none")


In [80]:
# create trainer
from transformers import Trainer

In [81]:
trainer3 =Trainer(model=lora_model,args=training_args,train_dataset=train_tokenized2,eval_dataset=val_tokenized2,
                 processing_class=tokenizer3,data_collator=data_collator,compute_metrics=compute_metrics)

In [82]:
# lets train Model 3

In [83]:
train_result1=trainer3.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.384397,0.376446,0.832281,0.849402,0.786716,0.816858
2,0.365347,0.324336,0.857544,0.844090,0.859041,0.851500
3,0.316600,0.317467,0.860000,0.856716,0.847232,0.851948


In [84]:
val_results=trainer3.evaluate()
print(val_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.316600,0.317467,3,0.860000,0.856716,0.847232,0.851948


{'eval_loss': 0.3174667954444885, 'eval_accuracy': 0.86, 'eval_precision': 0.8567164179104477, 'eval_recall': 0.8472324723247232, 'eval_f1': 0.8519480519480519}


In [85]:
test_results = trainer3.evaluate(
    eval_dataset=test_tokenized2
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.316600,0.324862,3,0.859348,0.851879,0.852507,0.852193


{'eval_loss': 0.32486221194267273, 'eval_accuracy': 0.8593475973342687, 'eval_precision': 0.8518791451731761, 'eval_recall': 0.8525073746312685, 'eval_f1': 0.8521931441208994}


In [86]:
pred_output = trainer3.predict(test_tokenized2)

In [87]:
logits = pred_output.predictions

y_true = pred_output.label_ids

y_pred = np.argmax(logits, axis=-1)

In [88]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Not Sarcastic", "Sarcastic"],
        digits=4
    )
)

               precision    recall  f1-score   support

Not Sarcastic     0.8661    0.8656    0.8658      1495
    Sarcastic     0.8519    0.8525    0.8522      1356

     accuracy                         0.8593      2851
    macro avg     0.8590    0.8590    0.8590      2851
 weighted avg     0.8594    0.8593    0.8593      2851



In [89]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

print(cm)

[[1294  201]
 [ 200 1156]]


In [99]:
import pandas as pd

# Transformer model comparison
comparison = pd.DataFrame({
    "Model": [
        "Model 1",
        "Model 2 ⭐ Recommended",
        "Model 3"
    ],

    "Approach": [
        "Frozen DistilBERT + Classifier Head",
        "Pre-fine-tuned Sarcasm Model (Evaluation Only)",
        "DistilBERT + LoRA (PEFT)"
    ],

    "Test Accuracy": [
        0.8169,
        0.9400,
        0.8593
    ],

    "Precision": [
        0.8268,
        0.9618,
        0.8561
    ],

    "Recall": [
        0.7780,
        0.9100,
        0.8466
    ],

    "F1 Score": [
        0.8017,
        0.9352,
        0.8513
    ],

    "Trainable Parameters": [
        "Classifier Head Only",
        "0 (Evaluation Only)",
        "739,586"
    ],

    "Trainable %": [
        "—",
        "0%",
        "1.09%"
    ]
})


# Highlight recommended model
def highlight_recommended(row):
    if row["Model"] == "Model 2 ⭐ Recommended":
        return ["background-color: lightgreen; font-weight: bold"] * len(row)
    return [""] * len(row)


# Display formatted comparison table
comparison.style \
    .format({
        "Test Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    }) \
    .apply(highlight_recommended, axis=1) \
    .set_caption("Transformer Models Comparison — Sarcasm Detection")

,Model,Approach,Test Accuracy,Precision,Recall,F1 Score,Trainable Parameters,Trainable %
0,Model 1,Frozen DistilBERT + Classifier Head,81.69%,82.68%,77.80%,80.17%,Classifier Head Only,—
1,Model 2 ⭐ Recommended,Pre-fine-tuned Sarcasm Model (Evaluation Only),94.00%,96.18%,91.00%,93.52%,0 (Evaluation Only),0%
2,Model 3,DistilBERT + LoRA (PEFT),85.93%,85.61%,84.66%,85.13%,"739,586",1.09%
